# 智能体
智能体将语言模型与工具结合，创建能够对任务进行推理、决定使用哪些工具并迭代地解决问题的系统。

`create_agent` 提供了一个生产就绪的智能体实现。

一个 LLM 智能体在循环中运行工具以实现目标。智能体一直运行直到满足停止条件——即模型发出最终输出或达到迭代限制。

## 1. 核心组件
### 1.1 模型
智能体的推理引擎是模型。它可以通过多种方式指定，支持静态和动态模型选择。
#### 1.1.1 静态模型
静态模型在创建智能体时配置一次，并在整个执行过程中保持不变。这是最常见和直接的方法。

要从模型标识符字符串初始化静态模型：

In [4]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3:0.6b",
    temperature=0.1,
    max_tokens=1000,
    timeout=30)

agent = create_agent(
    model,
    tools=[]
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
)

print(response)

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='c87e6a84-ac77-4226-849c-62a8a64f94cb'), AIMessage(content="I can't provide real-time weather information as I don't have access to live data. However, I can offer general weather information based on your location or ask you to specify your location for more accurate details. Let me know if you'd like to know the weather in a specific area!", additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-13T08:19:09.5362547Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5005643500, 'load_duration': 1007363100, 'prompt_eval_count': 16, 'prompt_eval_duration': 139158700, 'eval_count': 161, 'eval_duration': 3814617300, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'}, id='lc_run--019bb66f-e2d0-7ea3-8d85-8a7a9812dfde-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 161, 't

#### 1.1.2 动态模型
动态模型在运行时根据当前状态和上下文选择。这支持复杂的路由逻辑和成本优化。

要使用动态模型，使用 `@wrap_model_call` 装饰器创建中间件，该中间件会修改请求中的模型：

In [8]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOllama(model="qwen3:0.6b")
advanced_model = ChatOllama(model="qwen3:1.7b")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    request.override(model=model)
    return handler(request)

agent = create_agent(
    model=basic_model,  # Default model
    tools=[],
    middleware=[dynamic_model_selection]
)

agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
)

print(response)

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='c87e6a84-ac77-4226-849c-62a8a64f94cb'), AIMessage(content="I can't provide real-time weather information as I don't have access to live data. However, I can offer general weather information based on your location or ask you to specify your location for more accurate details. Let me know if you'd like to know the weather in a specific area!", additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-13T08:19:09.5362547Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5005643500, 'load_duration': 1007363100, 'prompt_eval_count': 16, 'prompt_eval_duration': 139158700, 'eval_count': 161, 'eval_duration': 3814617300, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'}, id='lc_run--019bb66f-e2d0-7ea3-8d85-8a7a9812dfde-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 161, 't

### 1.2 工具
工具赋予智能体执行操作的能力。智能体超越了简单的仅模型工具绑定，通过促进以下功能：
- 按顺序进行多次工具调用（由单个提示触发）
- 在适当情况下并行工具调用
- 根据先前结果进行动态工具选择
- 工具重试逻辑和错误处理
- 工具调用之间状态的持久化
### 1.2.1 定义工具
将工具列表传递给智能体。

In [9]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model, tools=[search, get_weather])

如果提供了一个空的工具列表，智能体将由一个不具备工具调用功能的单个 LLM 节点组成。
#### 1.2.2 工具错误处理
要自定义工具错误的处理方式，使用 `@wrap_tool_call` 装饰器创建中间件

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage
from langchain_ollama import ChatOllama

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[search, get_weather],
    middleware=[handle_tool_errors]
)

当工具失败时，智能体将返回一个包含自定义错误消息的 ToolMessage
```
[
    ...
    ToolMessage(
        content="Tool error: Please check your input and try again. (division by zero)",
        tool_call_id="..."
    ),
    ...
]
```
#### 1.2.3 ReAct 循环中的工具使用
智能体遵循 ReAct（“推理+行动”）模式，在简短的推理步骤和有针对性的工具调用之间交替进行，并将产生的观察结果馈送到后续决策中，直到它们能够提供最终答案。
### 1.3 系统提示
可以通过提供提示来塑造智能体处理任务的方式。`system_prompt` 参数可以作为字符串提供。

In [12]:
agent = create_agent(
    model,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

当未提供 system_prompt 时，智能体会直接从消息中推断其任务。
#### 1.3.1 动态系统提示
对于更高级的用例，当您需要根据运行时上下文或智能体状态修改系统提示时，可以使用中间件。

`@dynamic_prompt` 装饰器创建了一个中间件，可以根据模型请求动态生成系统提示：

In [19]:
from typing import TypedDict # 用于定义字典类型的结构化类型

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_ollama import ChatOllama

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "beginner"}
)

print(result)

{'messages': [HumanMessage(content='Explain machine learning', additional_kwargs={}, response_metadata={}, id='08b352f4-f83c-4cc8-a515-f20e3fe6673d'), AIMessage(content="Machine learning is a type of artificial intelligence that lets algorithms learn from data to make predictions or decisions. Here's a simple breakdown:\n\n1. **What is machine learning?**  \n   It's a process where a computer system improves its performance by learning patterns from data. Instead of being told the right way, the system learns from examples it sees.\n\n2. **Types of machine learning**:\n   - **Supervised learning**: The model learns from labeled data (like a teacher telling it what to do).  \n     Example: A doctor using patient data to predict disease outcomes.  \n   - **Unsupervised learning**: The model finds patterns in data without labels.  \n     Example: A company using customer data to group people into similar groups.  \n   - **Reinforcement learning**: The model learns by rewarding it to do so

## 2. 调用
可以通过向其 `State` 传递更新来调用智能体。所有智能体都在其状态中包含一系列消息；要调用智能体，请传递一条新消息

In [23]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]},
    context={"user_role": "user"}
)

## 3. 高级概念
### 3.1 结构化输出
在某些情况下，希望智能体以特定格式返回输出。LangChain 通过 response_format 参数提供结构化输出策略。
#### 3.1.1 工具策略 (ToolStrategy)
`ToolStrategy` 使用人工工具调用来生成结构化输出。这适用于任何支持工具调用的模型。

In [24]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    # tools=[search_tool],
    tools=[],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

#### 3.1.2 提供者策略 (ProviderStrategy)
`ProviderStrategy` 使用模型提供者原生的结构化输出生成。这更可靠，但仅适用于支持原生结构化输出的提供者（例如 OpenAI）。

In [31]:
# ProviderStrategy 不适用于 Ollama
from langchain.agents.structured_output import ProviderStrategy

agent = create_agent(
    model="gpt-4o",
    response_format=ProviderStrategy(ContactInfo)
)

### 3.2 记忆
智能体通过消息状态自动维护对话历史。还可以配置智能体使用自定义状态模式，以在对话过程中记住额外信息。

存储在状态中的信息可以被视为智能体的短期记忆：

自定义状态模式必须将 AgentState 作为 TypedDict 扩展。

定义自定义状态有两种方式：

- 通过中间件（首选）
- 通过 `state_schema` 在 `create_agent` 上

#### 3.2.1 通过中间件定义状态
当自定义状态需要被特定的中间件钩子和附加到该中间件的工具访问时，请使用中间件定义自定义状态。

In [34]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware


class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    # tools = [tool1, tool2]
    tools = []
    def before_model(self, state: CustomState, runtime) -> dict[str, any] | None:
        ...

agent = create_agent(
    model,
    # tools=tools,
    tools=[],
    middleware=[CustomMiddleware()]
)

# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

### 3.2.2 通过 state_schema 定义状态
使用 `state_schema` 参数作为定义仅在工具中使用的自定义状态的快捷方式。

In [33]:
from langchain.agents import AgentState


class CustomState(AgentState):
    user_preferences: dict

agent = create_agent(
    model,
    # tools=[tool1, tool2],
    tools=[],
    state_schema=CustomState
)
# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

### 3.3 流式处理
我们已经了解了如何使用 invoke 调用智能体以获取最终响应。如果智能体执行多个步骤，这可能需要一段时间。为了显示中间进度，我们可以流式传输消息。

In [35]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

Agent: Search for AI news and summarize the findings
Agent: To summarize AI news findings, here's a structured overview:

**Key Findings from Recent AI Developments:**  
1. **Algorithm Breakthroughs**: A new neural network model, "Nexus-2," achieved 98.7% accuracy in medical diagnosis tasks, outperforming existing methods by 12%.  
2. **Ethical Implications**: A report highlights concerns over AI bias in hiring systems, emphasizing the need for diverse datasets to mitigate biases.  
3. **Healthcare Applications**: AI-powered tools, such as predictive health models, are being integrated into clinics to improve early diagnosis rates, with some countries reporting 30% increases in patient outcomes.  

**Summary:**  
Recent AI advancements demonstrate rapid progress in areas like medical diagnostics and healthcare integration. While ethical concerns persist, the positive impact on efficiency and accuracy is notable. These developments underscore AI's potential to enhance productivity and a

### 3.3 中间件
中间件为在执行的不同阶段自定义智能体行为提供了强大的可扩展性。可以使用中间件来
- 在模型调用前处理状态（例如，消息截断、上下文注入）
- 修改或验证模型的响应（例如，防护措施、内容过滤）
- 使用自定义逻辑处理工具执行错误
- 根据状态或上下文实现动态模型选择
- 添加自定义日志、监控或分析

中间件无缝集成到智能体的执行图中，允许在关键点拦截和修改数据流，而无需更改核心智能体逻辑。